## Exercitiul 2 - Deschiderea unui restaurant fast-food

Clientii sosesc conform unui proces Poisson cu intensitatea:

$$
\lambda = 20 \text{ clienti/ora}
$$

Prin urmare, timpul dintre doua sosiri consecutive este exponential, cu media:

$$
\frac{60}{20}=3 \text{ minute}
$$

Timpul de servire la casa este exponential cu media de 2 minute, iar timpul de preparare este exponential cu media de $\alpha$ minute.

Sistemul contine doua cozi FIFO in serie:

1. clientul asteapta si este servit la casa;
2. dupa plata, comanda intra in coada statiei de gatit.

### a) Simularea unei ore

Pentru fiecare client generam:

- momentul sosirii;
- timpul necesar pentru plata;
- timpul necesar pentru preparare.

La casa, un client poate incepe plata doar dupa ce a sosit si dupa ce clientul precedent a terminat:

$$
startCasa_i =
\max(sosire_i,\ finalCasa_{i-1})
$$

Momentul terminarii platii este:

$$
finalCasa_i = startCasa_i + timpPlata_i
$$

La statia de gatit, comanda poate incepe sa fie preparata doar dupa terminarea platii si dupa terminarea comenzii precedente:

$$
startGatit_i =
\max(finalCasa_i,\ finalGatit_{i-1})
$$

iar:

$$
finalGatit_i =
startGatit_i+timpPreparare_i
$$

Timpul total petrecut de client in sistem este:

$$
T_i = finalGatit_i-sosire_i
$$

In [1]:
import numpy as np

def simuleaza_ora(alpha, rng):
    # Sosirile: timpul dintre sosiri ~ Exp(media 3 minute)
    sosiri = []

    t = rng.exponential(3)

    while t <= 60:
        sosiri.append(t)
        t += rng.exponential(3)

    sosiri = np.array(sosiri)
    n = len(sosiri)

    # Timpi de servire
    timp_plata = rng.exponential(2, n)
    timp_preparare = rng.exponential(alpha, n)

    final_casa = np.zeros(n)
    final_gatit = np.zeros(n)

    for i in range(n):
        # Coada de la casa
        if i == 0:
            start_casa = sosiri[i]
        else:
            start_casa = max(
                sosiri[i],
                final_casa[i - 1]
            )

        final_casa[i] = start_casa + timp_plata[i]

        # Coada de la gatit
        if i == 0:
            start_gatit = final_casa[i]
        else:
            start_gatit = max(
                final_casa[i],
                final_gatit[i - 1]
            )

        final_gatit[i] = (
            start_gatit + timp_preparare[i]
        )

    timp_total = final_gatit - sosiri

    timp_asteptare = (
        timp_total
        - timp_plata
        - timp_preparare
    )

    return timp_total, timp_asteptare

### b) Estimarea probabilitatii si cautarea lui alpha

Pentru o simulare consideram ca sistemul a avut succes daca fiecare client sosit in acea ora a primit mancarea in maximum 15 minute:

$$
\max_i T_i \leq 15
$$

Repetam simularea de mai multe ori. Probabilitatea este estimata prin frecventa relativa:

$$
\hat{P}
=
\frac{\text{numarul simularilor reusite}}
{\text{numarul total de simulari}}
$$

In [2]:
def estimeaza_probabilitate(alpha, nr_simulari=20000, seed=42):
    rng = np.random.default_rng(seed)

    reusite = 0

    for _ in range(nr_simulari):
        timp_total, _ = simuleaza_ora(alpha, rng)

        if np.all(timp_total <= 15):
            reusite += 1

    return reusite / nr_simulari

In [3]:
valori_alpha = [0.1, 0.25, 0.5, 1, 1.5, 2]

for alpha in valori_alpha:
    p = estimeaza_probabilitate(
        alpha,
        nr_simulari=20000,
        seed=42
    )

    print(
        f"alpha = {alpha:.2f}, "
        f"P = {p:.4f}"
    )

alpha = 0.10, P = 0.8073
alpha = 0.25, P = 0.8005
alpha = 0.50, P = 0.7861
alpha = 1.00, P = 0.7372
alpha = 1.50, P = 0.6243
alpha = 2.00, P = 0.4436


In [4]:
def simuleaza_ora_gatit_instant(rng):
    sosiri = []

    t = rng.exponential(3)

    while t <= 60:
        sosiri.append(t)
        t += rng.exponential(3)

    sosiri = np.array(sosiri)
    n = len(sosiri)

    timp_plata = rng.exponential(2, n)

    final_casa = np.zeros(n)

    for i in range(n):
        if i == 0:
            start_casa = sosiri[i]
        else:
            start_casa = max(
                sosiri[i],
                final_casa[i - 1]
            )

        final_casa[i] = (
            start_casa + timp_plata[i]
        )

    # Gatitul este instantaneu
    timp_total = final_casa - sosiri

    return timp_total

In [5]:
rng = np.random.default_rng(42)

nr_simulari = 100000
reusite = 0

for _ in range(nr_simulari):
    timp_total = simuleaza_ora_gatit_instant(rng)

    if np.all(timp_total <= 15):
        reusite += 1

p_maxim = reusite / nr_simulari

print("Probabilitate in cazul optim:", p_maxim)

Probabilitate in cazul optim: 0.80962


Chiar si in cazul limita in care timpul de preparare este zero, probabilitatea estimata ca toti clientii sa termine in cel mult 15 minute este de aproximativ:

$$
0.81
$$

Aceasta valoare este deja sub pragul cerut de:

$$
0.95
$$

Orice valoare $\alpha>0$ introduce timp suplimentar de preparare si poate doar sa mareasca timpii de finalizare.

Prin urmare, in conditiile modelului dat:

$$
\boxed{
\text{nu exista o valoare } \alpha>0
\text{ pentru care probabilitatea sa fie cel putin }95\%
}
$$

Limitarea provine deja din coada de la casa.

### c) Timpul mediu

La punctul b) am constatat ca nu exista nicio valoare $\alpha>0$ care sa satisfaca cerinta de probabilitate de cel putin $95\%$.

Prin urmare, nu exista o valoare valida a lui $\alpha$ pentru care sa putem calcula timpul mediu cerut la acest punct.

Putem totusi analiza cazul limita cu gatire instantanee pentru a vedea performanta maxima posibila a sistemului.

In [6]:
rng = np.random.default_rng(42)

nr_simulari = 100000

suma_timp = 0
suma_asteptare = 0
nr_clienti = 0

for _ in range(nr_simulari):
    sosiri = []

    t = rng.exponential(3)

    while t <= 60:
        sosiri.append(t)
        t += rng.exponential(3)

    sosiri = np.array(sosiri)
    n = len(sosiri)

    timp_plata = rng.exponential(2, n)

    final_casa = np.zeros(n)

    for i in range(n):
        if i == 0:
            start_casa = sosiri[i]
        else:
            start_casa = max(
                sosiri[i],
                final_casa[i - 1]
            )

        final_casa[i] = (
            start_casa + timp_plata[i]
        )

    timp_total = final_casa - sosiri
    timp_asteptare = timp_total - timp_plata

    suma_timp += timp_total.sum()
    suma_asteptare += timp_asteptare.sum()
    nr_clienti += n

print(
    "Timp mediu total:",
    suma_timp / nr_clienti
)

print(
    "Timp mediu de asteptare:",
    suma_asteptare / nr_clienti
)

Timp mediu total: 4.969405743701685
Timp mediu de asteptare: 2.969836539346578


### d) Detalii despre simulare

Am utilizat un generator aleator cu:

$$
seed=42
$$

pentru ca rezultatele sa poata fi reproduse.

Pentru explorarea valorilor lui $\alpha$ am folosit 20000 de simulari pentru fiecare valoare testata.

Pentru verificarea cazului limita cu gatire instantanee am folosit 100000 de simulari.

Initial am testat mai multe valori ale lui $\alpha$ pe o grila. Deoarece probabilitatea era deja mult sub $95\%$ pentru valori foarte mici ale lui $\alpha$, am verificat separat cazul limita $\alpha\to0$, in care timpul de preparare este considerat zero.

Chiar si in acest caz favorabil, probabilitatea estimata este de aproximativ $81\%$, deci pragul de $95\%$ nu poate fi atins in modelul dat.

Estimarea Monte Carlo poate varia usor de la o rulare la alta deoarece foloseste valori generate aleator. Cu cat numarul de simulari este mai mare, cu atat variatia estimarii este in general mai mica. Fixarea semintei aleatoare permite reproducerea aceluiasi rezultat.